# Setup

In [1]:
"""
Parsing Exploration: SEC 10-K Filing Structure

Goal: Take one full-submission.txt (the gnarly multi-document SGML blob)
and extract the primary 10-K HTML from it.
"""
from pathlib import Path
import re
from collections import Counter

# Notebook lives in notebooks/, so go up one to find project root
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "raw" / "sec-edgar-filings"

print(f"Project root: {PROJECT_ROOT}")
print(f"Data dir exists: {DATA_DIR.exists()}")
print(f"Tickers found: {sorted(d.name for d in DATA_DIR.iterdir() if d.is_dir())}")

Project root: c:\Users\User\Desktop\portfolio_projects\ML_projects\finintel
Data dir exists: True
Tickers found: ['AAPL', 'GOOGL', 'JPM', 'MSFT', 'TSLA']


In [2]:
# We'll start with the most recent AAPL 10-K — the one we already eyeballed yesterday.
ticker = "AAPL"
form = "10-K"

filing_dirs = sorted((DATA_DIR / ticker / form).iterdir())
latest_dir = filing_dirs[-1]
filing_path = latest_dir / "full-submission.txt"

size_mb = filing_path.stat().st_size / (1024 * 1024)
print(f"Filing: {ticker} {form}")
print(f"Accession: {latest_dir.name}")
print(f"Path: {filing_path.relative_to(PROJECT_ROOT)}")
print(f"Size: {size_mb:.2f} MB")

Filing: AAPL 10-K
Accession: 0000320193-25-000079
Path: data\raw\sec-edgar-filings\AAPL\10-K\0000320193-25-000079\full-submission.txt
Size: 8.96 MB


In [3]:
raw = filing_path.read_text(encoding="utf-8", errors="replace")
print(f"Total characters: {len(raw):,}")
print(f"\n--- First 800 chars (the SEC-HEADER block) ---\n")
print(raw[:800])

Total characters: 9,392,337

--- First 800 chars (the SEC-HEADER block) ---

<SEC-DOCUMENT>0000320193-25-000079.txt : 20251031
<SEC-HEADER>0000320193-25-000079.hdr.sgml : 20251031
<ACCEPTANCE-DATETIME>20251031060126
ACCESSION NUMBER:		0000320193-25-000079
CONFORMED SUBMISSION TYPE:	10-K
PUBLIC DOCUMENT COUNT:		91
CONFORMED PERIOD OF REPORT:	20250927
FILED AS OF DATE:		20251031
DATE AS OF CHANGE:		20251031

FILER:

	COMPANY DATA:	
		COMPANY CONFORMED NAME:			Apple Inc.
		CENTRAL INDEX KEY:			0000320193
		STANDARD INDUSTRIAL CLASSIFICATION:	ELECTRONIC COMPUTERS [3571]
		ORGANIZATION NAME:           	06 Technology
		EIN:				942404110
		STATE OF INCORPORATION:			CA
		FISCAL YEAR END:			0927

	FILING VALUES:
		FORM TYPE:		10-K
		SEC ACT:		1934 Act
		SEC FILE NUMBER:	001-36743
		FILM NUMBER:		251437791

	BUSINESS ADDRESS:	
		STREET 1:		ONE APPLE PARK WAY
		CITY:			CUPERT


In [4]:
doc_starts = list(re.finditer(r'<DOCUMENT>', raw))
doc_ends = list(re.finditer(r'</DOCUMENT>', raw))

print(f"Opening <DOCUMENT> tags: {len(doc_starts)}")
print(f"Closing </DOCUMENT> tags: {len(doc_ends)}")
# These should match each other AND the PUBLIC DOCUMENT COUNT in the header

Opening <DOCUMENT> tags: 90
Closing </DOCUMENT> tags: 90


In [5]:
type_pattern = re.compile(r'<DOCUMENT>\s*<TYPE>([^\n<]+)')
types = [t.strip() for t in type_pattern.findall(raw)]

print(f"Total documents: {len(types)}\n")
print("Top document types in this filing:")
for doc_type, count in Counter(types).most_common(15):
    print(f"  {count:3d}  {doc_type}")

Total documents: 90

Top document types in this filing:
   74  XML
    2  GRAPHIC
    1  10-K
    1  EX-4.1
    1  EX-21.1
    1  EX-23.1
    1  EX-31.1
    1  EX-31.2
    1  EX-32.1
    1  EX-101.SCH
    1  EX-101.CAL
    1  EX-101.DEF
    1  EX-101.LAB
    1  EX-101.PRE
    1  JSON


In [6]:
# Regex breakdown:
#   <DOCUMENT>          - block opens
#   <TYPE>...           - capture filing type ("10-K")
#   <SEQUENCE>...       - capture sequence number
#   <FILENAME>...       - capture original HTML filename (e.g., aapl-20250927.htm)
#   <DESCRIPTION>...    - optional human description
#   <TEXT>...</TEXT>    - the actual HTML payload (DOTALL so it spans newlines)
#   </DOCUMENT>         - block closes
doc_pattern = re.compile(
    r'<DOCUMENT>\s*'
    r'<TYPE>([^\n<]+)\s*'
    r'<SEQUENCE>([^\n<]+)\s*'
    r'<FILENAME>([^\n<]+)\s*'
    r'(?:<DESCRIPTION>[^\n<]*\s*)?'  # ?: = non-capturing; this group is optional
    r'<TEXT>(.*?)</TEXT>\s*'
    r'</DOCUMENT>',
    re.DOTALL,
)

documents = doc_pattern.findall(raw)
print(f"Successfully parsed {len(documents)} documents (out of {len(types)} expected)")

# Filter to just the primary filing
primary = [d for d in documents if d[0].strip() == form]
print(f"Documents of type '{form}': {len(primary)}")

doc_type, sequence, filename, html_payload = primary[0]
print(f"\n--- Primary 10-K document ---")
print(f"  Type:     {doc_type.strip()}")
print(f"  Sequence: {sequence.strip()}")
print(f"  Filename: {filename.strip()}")
print(f"  HTML size: {len(html_payload):,} chars (~{len(html_payload)/1024:.0f} KB)")

Successfully parsed 90 documents (out of 90 expected)
Documents of type '10-K': 1

--- Primary 10-K document ---
  Type:     10-K
  Sequence: 1
  Filename: aapl-20250927.htm
  HTML size: 1,520,224 chars (~1485 KB)


In [7]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / ticker / form / latest_dir.name
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

primary_html_path = PROCESSED_DIR / "primary.html"
primary_html_path.write_text(html_payload, encoding="utf-8")

print(f"Saved to: {primary_html_path.relative_to(PROJECT_ROOT)}")
print(f"\nOpen it in your browser:")
print(f"  file:///{primary_html_path.as_posix()}")

Saved to: data\processed\AAPL\10-K\0000320193-25-000079\primary.html

Open it in your browser:
  file:///c:/Users/User/Desktop/portfolio_projects/ML_projects/finintel/data/processed/AAPL/10-K/0000320193-25-000079/primary.html
